# Elastic Net sensitivity analysis

Evaluate Elastic Net Logistic Regression across the five stratified partitions and compare it with the matched L2-regularized baseline.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_FILE = PROJECT_ROOT / "data" / "gallstone.xlsx"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Download the UCI Gallstone dataset and save it to {DATA_FILE}")


In [ ]:
import pandas as pd
import numpy as np
df = pd.read_excel(DATA_FILE)
print('Dataset shape:', df.shape)


In [ ]:
"""Elastic Net Logistic Regression and matched L2 baseline."""
import json, platform, sys, time, warnings
from collections import defaultdict
import numpy as np
import sklearn
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, balanced_accuracy_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.exceptions import ConvergenceWarning

SEEDS=[42, 52, 62, 72, 82]
FEAT_GROUPS={
'demographic':['Age','Gender','Height','Weight','Body Mass Index (BMI)'],
'clinical':['Comorbidity','Coronary Artery Disease (CAD)','Hypothyroidism','Hyperlipidemia','Diabetes Mellitus (DM)','Hepatic Fat Accumulation (HFA)'],
'bioimpedance':['Total Body Water (TBW)','Extracellular Water (ECW)','Intracellular Water (ICW)','Extracellular Fluid/Total Body Water (ECF/TBW)','Total Body Fat Ratio (TBFR) (%)','Lean Mass (LM) (%)','Body Protein Content (Protein) (%)','Visceral Fat Rating (VFR)','Bone Mass (BM)','Muscle Mass (MM)','Obesity (%)','Total Fat Content (TFC)','Visceral Fat Area (VFA)','Visceral Muscle Area (VMA) (Kg)'],
'laboratory':['Glucose','Total Cholesterol (TC)','Low Density Lipoprotein (LDL)','High Density Lipoprotein (HDL)','Triglyceride','Aspartat Aminotransferaz (AST)','Alanin Aminotransferaz (ALT)','Alkaline Phosphatase (ALP)','Creatinine','Glomerular Filtration Rate (GFR)','C-Reactive Protein (CRP)','Hemoglobin (HGB)','Vitamin D']
}
SCENARIOS={
'A_baseline':['demographic','clinical'],
'B_plus_bia':['demographic','clinical','bioimpedance'],
'C_plus_lab':['demographic','clinical','laboratory'],
'D_full':['demographic','clinical','bioimpedance','laboratory']
}
CONTRASTS={'B-A':('B_plus_bia','A_baseline'),'C-A':('C_plus_lab','A_baseline'),'D-B':('D_full','B_plus_bia'),'D-C':('D_full','C_plus_lab')}
METRICS=['accuracy','f1','roc_auc']
CS=[float(x) for x in np.logspace(-3,2,20)]
L1_RATIOS=[0.1,0.3,0.5,0.7,0.9]

def summarize(rows, groupby, fields):
    d=defaultdict(list)
    for r in rows:
        d[tuple(r[k] for k in groupby)].append(r)
    result=[]
    for key,sub in sorted(d.items()):
        record=dict(zip(groupby,key))
        record['n']=len(sub)
        for met in fields:
            v=np.array([float(q[met]) for q in sub])
            record['mean_'+met]=float(np.mean(v))
            record['sd_'+met]=float(np.std(v,ddof=1)) if len(v)>1 else None
            record['min_'+met]=float(np.min(v))
            record['max_'+met]=float(np.max(v))
        result.append(record)
    return result

def run_experiment(data, header, reference_lr=None, verbose=True):
    hlist=list(header)
    assert len(data)==319 and 'Gallstone Status' in hlist
    required=[c for group in FEAT_GROUPS.values() for c in group]
    if any(c not in hlist for c in required):
        raise ValueError('Dataset column mismatch: '+repr([c for c in required if c not in hlist]))
    if len(set(required))!=38: raise ValueError('Expected 38 unique feature names')
    raw=np.asarray(data,dtype=object)
    y=np.array([int(float(z)==0.0) for z in raw[:,hlist.index('Gallstone Status')]],dtype=int)
    if (np.unique(y,return_counts=True)[1].min() <100):
        raise ValueError('Target class mapping invalid')
    XX=np.array([[float(r[hlist.index(f)]) for f in required] for r in raw],dtype=float)
    if not np.all(np.isfinite(XX)):
        raise ValueError('Missing, infinite or nonnumeric feature values')
    pos={n:i for i,n in enumerate(required)}
    runs=[]; tests=[]; folds=[]; splits=[]; all_idx=np.arange(len(y))
    for seed in SEEDS:
        tr,te=train_test_split(all_idx,test_size=.20,stratify=y,random_state=seed)
        cv=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=seed).split(np.zeros(len(tr)),y[tr]))
        for subset, ids in [('train',tr),('test',te)]:
            splits.extend({'split_seed':seed,'subset':subset,'dataset_row_index':int(idx)} for idx in ids)
        for fold,(itr,iva) in enumerate(cv,1):
            folds.extend({'split_seed':seed,'fold':fold,'subset':subset,'dataset_row_index':int(tr[j])} for subset,sub in [('train',itr),('validation',iva)] for j in sub)
        for scenario, groups in SCENARIOS.items():
            features=[f for g in groups for f in FEAT_GROUPS[g]]
            j=[pos[f] for f in features]
            xtr=XX[tr][:,j]; xte=XX[te][:,j]
            pipe=Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(
                penalty='elasticnet',solver='saga',max_iter=4000,tol=1e-3,random_state=seed))])
            grid=GridSearchCV(pipe, {'clf__C':CS,'clf__l1_ratio':L1_RATIOS},cv=cv,
                              scoring='accuracy',n_jobs=2,refit=True,error_score='raise')
            start=time.perf_counter()
            with warnings.catch_warnings(record=True) as ww:
                warnings.filterwarnings('ignore',category=FutureWarning)
                warnings.simplefilter('always',ConvergenceWarning)
                grid.fit(xtr,y[tr])
                n_warn=sum(issubclass(w.category,ConvergenceWarning) for w in ww)
            fit_seconds=time.perf_counter()-start
            pred=grid.predict(xte)
            prob=grid.predict_proba(xte)[:,1]
            tn,fp,fn,tp=confusion_matrix(y[te],pred,labels=[0,1]).ravel()
            cv_idx=int(grid.best_index_)
            acc=float(accuracy_score(y[te],pred)); f1=float(f1_score(y[te],pred))
            roc=float(roc_auc_score(y[te],prob))
            best_C=float(grid.best_params_['clf__C']); best_l1=float(grid.best_params_['clf__l1_ratio'])
            rec={'split_seed':seed,'optimizer':'GridSearch','model':'ElasticNetLR',
                 'scenario':scenario,'n_features':len(features),'train_n':len(tr),'test_n':len(te),
                 'cv_best_accuracy':float(grid.best_score_),
                 'cv_accuracy_sd':float(grid.cv_results_['std_test_score'][cv_idx]),
                 'accuracy':acc,'precision':float(precision_score(y[te],pred,zero_division=0)),
                 'recall':float(recall_score(y[te],pred,zero_division=0)),
                 'specificity':float(tn/(tn+fp)),
                 'balanced_accuracy':float(balanced_accuracy_score(y[te],pred)),
                 'f1':f1,'roc_auc':roc,'mcc':float(matthews_corrcoef(y[te],pred)),
                 'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),
                 'best_C':best_C,'best_l1_ratio':best_l1,'cv_candidates':100,
                 'cv_folds':5,'fit_seconds':float(fit_seconds),
                 'convergence_warnings':int(n_warn),'best_params_json':json.dumps({'C':best_C,'l1_ratio':best_l1})}
            runs.append(rec)
            tests.extend({'split_seed':seed,'model':'ElasticNetLR','scenario':scenario,
                          'dataset_row_index':int(i),'y_true':int(t),'y_pred':int(p),'prob_positive':float(q)}
                         for i,t,p,q in zip(te,y[te],pred,prob))
            if verbose: print(f'Seed {seed}, {scenario}: CV={rec["cv_best_accuracy"]:.3f}; test Acc={acc:.3f}; AUC={roc:.3f}; time={fit_seconds:.1f}s',flush=True)
    lookup={(r['split_seed'],r['scenario']):r for r in runs}
    contrast_rows=[]
    for seed in SEEDS:
        for contrast,(higher,lower) in CONTRASTS.items():
            for metric in METRICS:
                a=lookup[seed,higher][metric];b=lookup[seed,lower][metric]
                contrast_rows.append({'split_seed':seed,'model':'ElasticNetLR','contrast':contrast,'metric':metric,'higher':float(a),'reference':float(b),'delta':float(a-b)})
    contr_sum=summarize(contrast_rows,['contrast','metric'],['delta'])
    for s in contr_sum:
        group=[r for r in contrast_rows if r['contrast']==s['contrast'] and r['metric']==s['metric']]
        s['n_positive']=sum(r['delta']>1e-12 for r in group)
        s['n_zero']=sum(abs(r['delta'])<=1e-12 for r in group)
        s['n_negative']=sum(r['delta']< -1e-12 for r in group)
    comp=[]
    if reference_lr:
        ref={(int(r['split_seed']),str(r['scenario'])):r for r in reference_lr if str(r.get('model'))=='LR' and str(r.get('optimizer'))=='GridSearch'}
        if len(ref)!=20: raise ValueError('Original reference should contain exactly 20 GridSearch LR runs, got '+str(len(ref)))
        for r in runs:
            orig=ref[r['split_seed'],r['scenario']]
            for met in METRICS:
                comp.append({'split_seed':r['split_seed'],'scenario':r['scenario'],'metric':met,
                             'ElasticNetLR':float(r[met]),'OriginalLR':float(orig[met]),
                             'Delta_ElasticNet_minus_LR':float(r[met]-float(orig[met]))})
    comp_sum=summarize(comp,['scenario','metric'],['Delta_ElasticNet_minus_LR']) if comp else []
    config=[('purpose','Elastic Net Logistic Regression supplementary baseline for R1-9'),
            ('dataset','UCI Gallstone; same 319 observations and 38 original predictors'),
            ('positive label','raw 0 = gallstones present -> binary 1'),
            ('split_seeds',','.join(map(str,SEEDS))),
            ('outer split','5 repeated stratified 80:20 partitions; overlaps across splits'),
            ('inner CV','Fixed stratified 5-fold for each seed; shuffled with same seed'),
            ('preprocessing','StandardScaler fitted inside each inner fold; all predictors scaled as original'),
            ('search type','GridSearchCV; 20 logarithmic C values 0.001 to 100 x 5 l1_ratio values'),
            ('C_values',json.dumps(CS)),('l1_ratio_values',json.dumps(L1_RATIOS)),
            ('penalty','Elastic Net Logistic Regression; saga; tolerance 1e-3; max_iter 4000'),
            ('HPO fitness','mean inner CV accuracy; held-out labels never used in selection'),
            ('test metrics','Accuracy, precision, recall/sensitivity, specificity, balanced accuracy, F1, ROC-AUC, MCC'),
            ('comparison','Compare ElasticNetLR vs original GridSearch Logistic Regression for same outer seeds and feature scenario'),
            ('uncertainty','SD over overlapping splits is descriptive only, not a confidence interval'),
            ('python',platform.python_version()),('numpy',np.__version__),('scikit-learn',sklearn.__version__),
            ('software_platform',platform.platform())]
    return {'RunMetrics':runs,'ScenarioContrasts':contrast_rows,'ContrastSummary':contr_sum,
            'LRComparison':comp,'LRCompareSummary':comp_sum,'TestPredictions':tests,
            'SplitIndices':splits,'FoldIndices':folds,
            'Settings':[{'setting':k,'value':str(v)} for k,v in config]}


In [ ]:


import warnings, time
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def run_original_lr_reference(dataframe):
    header=list(dataframe.columns)
    raw=dataframe.to_numpy(dtype=object)
    columns=[f for group in FEAT_GROUPS.values() for f in group]
    X=np.array([[float(row[header.index(f)]) for f in columns] for row in raw],float)
    y=np.array([int(float(row[header.index("Gallstone Status")])==0.0) for row in raw],dtype=int)
    pos={f:i for i,f in enumerate(columns)}
    references=[]
    for seed in SEEDS:
        train,test=train_test_split(np.arange(len(y)),stratify=y,test_size=.2,random_state=seed)
        cv=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=seed).split(np.zeros(len(train)),y[train]))
        for scenario, groups in SCENARIOS.items():
            features=[f for g in groups for f in FEAT_GROUPS[g]]
            jj=[pos[x] for x in features]
            pipe=Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(
                solver='lbfgs',max_iter=2000,random_state=42))])
            search=GridSearchCV(pipe,{'clf__C':np.logspace(-3,2,100)},scoring='accuracy',
                                cv=cv,n_jobs=2,refit=True,error_score='raise')
            search.fit(X[train][:,jj],y[train])
            p=search.predict(X[test][:,jj]);q=search.predict_proba(X[test][:,jj])[:,1]
            references.append({'split_seed':seed,'scenario':scenario,'model':'LR','optimizer':'GridSearch',
                              'accuracy':float(accuracy_score(y[test],p)),
                              'f1':float(f1_score(y[test],p)),
                              'roc_auc':float(roc_auc_score(y[test],q))})
    return references

original_lr=run_original_lr_reference(df)
print('L2 Logistic Regression reference runs:', len(original_lr))


In [ ]:
all_results=run_experiment(df.to_numpy(dtype=object), list(df.columns),reference_lr=original_lr)
print("Elastic Net run:",len(all_results["RunMetrics"]),
      "| Paired contrasts:",len(all_results["ScenarioContrasts"]),
      "| LR comparisons:",len(all_results["LRComparison"]))


In [ ]:
from pathlib import Path
from importlib.metadata import version
output = RESULTS_DIR / "elastic_net.xlsx"
with pd.ExcelWriter(output,engine="openpyxl") as writer:
    for name, rows in all_results.items():
        pd.DataFrame(rows).to_excel(writer,sheet_name=name,index=False)
print('Saved:', output,
      "| size",round(output.stat().st_size/1024),"KB")
